### Calculation of Carbon contract for Difference (CCfD)-price

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [19]:
## Manually Input varibles 
output_file_name = 'CO2_prices.xlsx'

# fossil based methanol data
energy_density = 19.7*10**9                                 # J/t  (source: https://ebookcentral.proquest.com/lib/kbhnhh-ebooks/detail.action?docID=5116604)   
co2_indstr_high = 2.863                                     # kg*CO2/kg*MeOH   (source: https://www.sciencedirect.com/science/article/pii/S0360319922002415)
co2_indstr_low = 2.05                                       # kg*CO2/kg*MeOH   (source: https://www.methanol.org/wp-content/uploads/2022/01/CARBON-FOOTPRINT-OF-METHANOL-PAPER_1-31-22.pdf)
co2_indstr_perJ = 103 * 10**(-9)                            # kg*CO2/J  (source: https://www.methanol.org/wp-content/uploads/2022/01/CARBON-FOOTPRINT-OF-METHANOL-PAPER_1-31-22.pdf)

# Average emissions of grid power
avg_elctr_emissions_2045_DK = 0                             # kg Co2/(Wh electricity)  
avg_elctr_emissions_2035_DK = 96.0/(10**6)                  # kg Co2/(Wh electricity)  
avg_elctr_emissions_sq_DK = 499.8/(10**6)                   # kg Co2/(Wh electricity)  (source: https://energinet.dk/media/1duf3x0o/19_07249-24-generel-eldeklaration-historik-10536640_5986882_0.xlsx)

avg_elctr_emissions_2045_DE = 0                             # kg Co2/(Wh electricity)  
avg_elctr_emissions_2035_DE = 150 /(10**6)                  # kg Co2/(Wh electricity)  
avg_elctr_emissions_sq_DE = 363 /(10**6)                    # kg Co2/(Wh electricity)  (source: https://energinet.dk/media/1duf3x0o/19_07249-24-generel-eldeklaration-historik-10536640_5986882_0.xlsx)


# Power distribution within the hub for years 2019-2023
PPA_PV_sq = 0                                               # MWh (own result)
El_from_grid_sq_DE = 27811.3742                             # MWh (own result)
El_from_grid_sq_DK = 542058.5474                            # MWh (own result)
PPA_Wind_sq = 1122.197396                                   # MWh (own result)

PPA_PV_2035 = 0                                             # MWh (own result)
El_from_grid_2035_DE = 32094.84207                          # MWh (own result)
El_from_grid_2035_DK = 1295451.841                          # MWh (own result)
PPA_Wind_2035 = 72716.20941                                 # MWh (own result)

PPA_PV_2045 = 0                                             # MWh (own result)
El_from_grid_2045_DE = 345727.7922                          # MWh (own result)
El_from_grid_2045_DK = 3037386.059                          # MWh (own result)
PPA_Wind_2045 = 92708.06948                                 # MWh (own result)

# Yearly methanol demand in hub
demand_sq_yearly_t = 52350.15                               # t/a    (own result)
demand_2035_yearly_t = 132445.63                            # t/a    (own result)
demand_2045_yearly_t = 326093.89                            # t/a    (own result)

# Costs in €/t
costs_hub_sq = 1037.83                                      # €/t    (own result)
costs_hub_2035 = 1041.32                                    # €/t    (own result)
costs_hub_2045 = 968.65                                     # €/t    (own result)

costs_hub_rev_sq = 1376.143824                              # €/t    (own result)
costs_hub_rev_2035 = 1333.914886                            # €/t    (own result)
costs_hub_rev_2045 = 1248.741849                            # €/t    (own result)

# Costs industrial methanol 2019-2023   
costs_indstr_methanol2019 = 370                          # $/t    (source: https://www.irena.org/-/media/Files/IRENA/Agency/Publication/2021/Jan/IRENA_Innovation_Renewable_Methanol_2021.pdf)
costs_indstr_methanol2020 = 205                          # $/t    (source: https://www.irena.org/-/media/Files/IRENA/Agency/Publication/2021/Jan/IRENA_Innovation_Renewable_Methanol_2021.pdf)
costs_indstr_methanol2021 = 395                          # $/t    (source: https://shipandbunker.com/prices/emea/nwe/nl-rtm-rotterdam)
costs_indstr_methanol2022 = 399                          # $/t    (source: https://shipandbunker.com/prices/emea/nwe/nl-rtm-rotterdam)
costs_indstr_methanolsq = 321.5                          # $/t    (source: https://shipandbunker.com/prices/emea/nwe/nl-rtm-rotterdam)
exchange_rate2019 = 0.8932                               # €/$    (source: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)
exchange_rate2020 = 0.8602                               # €/$    (source: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)
exchange_rate2021 = 0.8945                               # €/$    (source: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)
exchange_rate2022 = 0.9371                               # €/$    (source: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)
exchange_ratesq = 0.9248                                 # €/$    (source: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/euro_reference_exchange_rates/html/eurofxref-graph-usd.en.html)


# European allowances 2019-2023
avg_EUA_2019 = 24.723125                                 # €/(t*CO2-eq)  (source: https://icapcarbonaction.com/en/ets-prices)
avg_EUA_2020 = 24.38660287                               # €/(t*CO2-eq)  (source: https://icapcarbonaction.com/en/ets-prices)
avg_EUA_2021 = 54.15156951                               # €/(t*CO2-eq)  (source: https://icapcarbonaction.com/en/ets-prices)
avg_EUA_2022 = 80.18404545                               # €/(t*CO2-eq)  (source: https://icapcarbonaction.com/en/ets-prices)
avg_EUA_sq = 83.59650224                                 # €/(t*CO2-eq)  (source: https://icapcarbonaction.com/en/ets-prices)

EUA_2035_dkk = 999 
EUA_2045_dkk = 2061                                      # DKK/(t*CO2-eq) (source: https://fm.dk/media/4tkj2o4w/noegletalskatalog_juni-2024.pdf)
exchange_rate = 0.1338
EUA_2035_EUR = EUA_2035_dkk*exchange_rate
EUA_2045_EUR = EUA_2045_dkk*exchange_rate
avg_EUA_2035 = EUA_2035_EUR
avg_EUA_2045 = EUA_2045_EUR

In [20]:
## Further calculations

# Co2 equivalent fossil based methanol production 
co2_industr_perWh = co2_indstr_perJ*3600                                             # kg*CO2/Wh MeOH

# Emissions per Wh
avg_emissions_indstr_methanol = co2_industr_perWh                                    # kg*CO2/Wh MeOH

# Electricity used for production, PV share and used grid electricity in production 
#SQ
#El_used_sq = El_from_grid_sq + PPA_PV_sq + PPA_Wind_sq                 # MWh 
el_demand_grid_sq = El_from_grid_sq_DE+ El_from_grid_sq_DK                     # MWh

#2035
#El_used_2035 = El_from_grid_2035+PPA_PV_2035 + PPA_Wind_2035                 # MWh 
el_demand_grid_2035 = El_from_grid_2035_DE + El_from_grid_2035_DK                           # MWh

#2045
#El_used_2045 = El_from_grid_2045+PPA_PV_2045 + PPA_Wind_2045                 # MWh 
el_demand_grid_2045 = El_from_grid_2045_DE + El_from_grid_2045_DK                           # MWh

# Grid electricity taken from the grid (Wh)

grid_electricity_sq_DK = El_from_grid_sq_DK * 10**6
grid_electricity_sq_DE = El_from_grid_sq_DE * 10**6

grid_electricity_2035_DK = El_from_grid_2035_DK * 10**6
grid_electricity_2035_DE = El_from_grid_2035_DE * 10**6

grid_electricity_2045_DK = El_from_grid_2045_DK * 10**6
grid_electricity_2045_DE = El_from_grid_2045_DE * 10**6


# Methanol demand  in hub (yearly, in Wh) 
demand_sq_yearly_Wh = (energy_density*demand_sq_yearly_t)/3600                              # Wh/a
demand_2035_yearly_Wh = (energy_density*demand_2035_yearly_t)/3600                              # Wh/a
demand_2045_yearly_Wh = (energy_density*demand_2045_yearly_t)/3600                              # Wh/a

# Emissions per year (t CO2)
emissions_hub_sq_DK = grid_electricity_sq_DK * avg_elctr_emissions_sq_DK / 1000
emissions_hub_sq_DE = grid_electricity_sq_DE * avg_elctr_emissions_sq_DE / 1000

emissions_hub_2035_DK = grid_electricity_2035_DK * avg_elctr_emissions_2035_DK / 1000
emissions_hub_2035_DE = grid_electricity_2035_DE * avg_elctr_emissions_2035_DE / 1000

emissions_hub_2045_DK = grid_electricity_2045_DK * avg_elctr_emissions_2045_DK / 1000
emissions_hub_2045_DE = grid_electricity_2045_DE * avg_elctr_emissions_2045_DE / 1000

# Total emissions per scenario (sum DK + DE)

emissions_hub_sq = emissions_hub_sq_DK + emissions_hub_sq_DE
emissions_hub_2035 = emissions_hub_2035_DK + emissions_hub_2035_DE
emissions_hub_2045 = emissions_hub_2045_DK + emissions_hub_2045_DE

# Industrial (fossil) methanol emissions per scenario (t CO2 / year)

emissions_indstrM_low_sq   = demand_sq_yearly_t   * co2_indstr_low
emissions_indstrM_high_sq  = demand_sq_yearly_t   * co2_indstr_high

emissions_indstrM_low_2035 = demand_2035_yearly_t * co2_indstr_low
emissions_indstrM_high_2035 = demand_2035_yearly_t * co2_indstr_high

emissions_indstrM_low_2045 = demand_2045_yearly_t * co2_indstr_low
emissions_indstrM_high_2045 = demand_2045_yearly_t * co2_indstr_high

# Emissions per tonne Methanol
emissions_per_tonne_sq= emissions_hub_sq/demand_sq_yearly_t                         # t*Co2/t*MeOH
emissions_per_tonne_2035 = emissions_hub_2035/demand_2035_yearly_t                         # t*Co2/t*MeOH
emissions_per_tonne_2045 = emissions_hub_2045/demand_2045_yearly_t                         # t*Co2/t*MeOH

# Costs for yearly demand
#Base
ym_costs_sq = costs_hub_sq*demand_sq_yearly_t                                        # €
ym_costs_2035 = costs_hub_2035*demand_2035_yearly_t                                        # €
ym_costs_2045 = costs_hub_2045*demand_2045_yearly_t                                        # €
#without revenue
ym_costs_sq_PV = costs_hub_rev_sq*demand_sq_yearly_t                                  # €
ym_costs_2035_PV = costs_hub_rev_2035*demand_2035_yearly_t                                  # €
ym_costs_2045_PV = costs_hub_rev_2045*demand_2045_yearly_t                                  # €

#conventional

ym_costs_indstrsq = costs_indstr_methanolsq*exchange_ratesq*demand_sq_yearly_t     # €
ym_costs_indstr2035 = costs_indstr_methanolsq*exchange_ratesq*demand_2035_yearly_t                                             # €
ym_costs_indstr2045 = costs_indstr_methanolsq*exchange_ratesq*demand_2045_yearly_t                                            # €

In [21]:
emissions_indstrM_low_sq - emissions_hub_sq

-173698.58332512004

In [22]:
ym_costs_sq-ym_costs_indstrsq

38765642.05602

In [23]:
emissions_indstrM_low_2045 - emissions_hub_2045

668492.4745

In [24]:
# Calculations of neccessary carbon prices   (extra costs per saved tonne of co2)
# BASE

# Comparison with lower CO2 emissions of industrial methanol
CPsql = (ym_costs_sq - ym_costs_indstrsq) / (emissions_indstrM_low_sq - emissions_hub_sq)
CP2035l = (ym_costs_2035 - ym_costs_indstr2035) / (emissions_indstrM_low_2035 - emissions_hub_2035)
CP2045l = (ym_costs_2045 - ym_costs_indstr2045) / (emissions_indstrM_low_2045 - emissions_hub_2045)

# Comparison with higher CO2 emissions of industrial methanol 
CPsqh = (ym_costs_sq - ym_costs_indstrsq) / (emissions_indstrM_high_sq - emissions_hub_sq)
CP2035h = (ym_costs_2035 - ym_costs_indstr2035) / (emissions_indstrM_high_2035 - emissions_hub_2035)
CP2045h = (ym_costs_2045 - ym_costs_indstr2045) / (emissions_indstrM_high_2045 - emissions_hub_2045)

# Purely green E-Methanol (lower industrial CO2)
CPsql_green = (ym_costs_sq - ym_costs_indstrsq) / emissions_indstrM_low_sq
CP2035l_green = (ym_costs_2035 - ym_costs_indstr2035) / emissions_indstrM_low_2035
CP2045l_green = (ym_costs_2045 - ym_costs_indstr2045) / emissions_indstrM_low_2045

# Purely green E-Methanol (higher industrial CO2)
CPsqh_green = (ym_costs_sq - ym_costs_indstrsq) / emissions_indstrM_high_sq
CP2035h_green = (ym_costs_2035 - ym_costs_indstr2035) / emissions_indstrM_high_2035
CP2045h_green = (ym_costs_2045 - ym_costs_indstr2045) / emissions_indstrM_high_2045


In [25]:
print("Carbon price compared to methanol with lower emission:", CPsql,CP2035l,CP2045l,)
print("Carbon price compared to methanol with higher emission:", CPsqh,CP2035h,CP2045h)
print("Carbon price compared to methanol with lower emission (green case):", CPsql_green, CP2035l_green,CP2045l_green)
print("Carbon price compared to methanol with higher emission (green_case):", CPsqh_green, CP2035h_green,CP2045h_green,)


print("CO2-equivalents:",emissions_per_tonne_sq,emissions_per_tonne_2035,emissions_per_tonne_2045,co2_indstr_low,co2_indstr_high)

Carbon price compared to methanol with lower emission: -223.17765242483568 692.2996817572953 327.47648780487805
Carbon price compared to methanol with higher emission: -295.6097260473432 394.134056568254 234.48368843870065
Carbon price compared to methanol with lower emission (green case): 361.2228292682927 362.925268292683 327.47648780487805
Carbon price compared to methanol with higher emission (green_case): 258.64715333566187 259.8661543835138 234.48368843870065
CO2-equivalents: 5.368015007122616 0.9753255207174446 0.0 2.05 2.863


In [26]:
# Create a dictionary to store the data
data_without_PV = {
    "Year": ["sq", "2035", "2045"],
    "Carbon Price Lower Emission": [CPsql, CP2035l, CP2045l],
    "Carbon Price Higher Emission": [CPsqh, CP2035h, CP2045h,],
    "Carbon Price Lower Emission (Green Case)": [CPsql_green, CP2035l_green, CP2045l_green],
    "Carbon Price Higher Emission (Green Case)": [CPsqh_green, CP2035h_green, CP2045h_green],
    "CO2 Equivalents": [emissions_per_tonne_sq, emissions_per_tonne_2035, emissions_per_tonne_2045],
    "CO2 Price [EUA]": [avg_EUA_sq, avg_EUA_2035, avg_EUA_2045 ]
}

# Convert the dictionary into a pandas DataFrame
df_values_without_PV = pd.DataFrame(data_without_PV)
df_values_without_PV

,Year,Carbon Price Lower Emission,Carbon Price Higher Emission,Carbon Price Lower Emission (Green Case),Carbon Price Higher Emission (Green Case),CO2 Equivalents,CO2 Price [EUA]
0,sq,-223.177652,-295.609726,361.222829,258.647153,5.368015,83.596502
1,2035,692.299682,394.134057,362.925268,259.866154,0.975326,133.666200
2,2045,327.476488,234.483688,327.476488,234.483688,0.000000,275.761800


In [27]:
# Calculations of neccessary carbon prices   (extra costs per saved tonne of co2)
# without revenues

# Comparison with lower CO2 emissions of industrial methanol
CPsql_PV = (ym_costs_sq_PV - ym_costs_indstrsq) / (emissions_indstrM_low_sq - emissions_hub_sq)
CP2035l_PV = (ym_costs_2035_PV - ym_costs_indstr2035) / (emissions_indstrM_low_2035 - emissions_hub_2035)
CP2045l_PV = (ym_costs_2045_PV - ym_costs_indstr2045) / (emissions_indstrM_low_2045 - emissions_hub_2045)

# Comparison with higher CO2 emissions of industrial methanol 
CPsqh_PV = (ym_costs_sq_PV - ym_costs_indstrsq) / (emissions_indstrM_high_sq - emissions_hub_sq)
CP2035h_PV = (ym_costs_2035_PV - ym_costs_indstr2035) / (emissions_indstrM_high_2035 - emissions_hub_2035)
CP2045h_PV = (ym_costs_2045_PV - ym_costs_indstr2045) / (emissions_indstrM_high_2045 - emissions_hub_2045)

# Purely green E-Methanol (lower industrial CO2)
CPsql_green_PV = (ym_costs_sq_PV - ym_costs_indstrsq) / emissions_indstrM_low_sq
CP2035l_green_PV = (ym_costs_2035_PV - ym_costs_indstr2035) / emissions_indstrM_low_2035
CP2045l_green_PV = (ym_costs_2045_PV - ym_costs_indstr2045) / emissions_indstrM_low_2045

# Purely green E-Methanol (higher industrial CO2)
CPsqh_green_PV = (ym_costs_sq_PV - ym_costs_indstrsq) / emissions_indstrM_high_sq
CP2035h_green_PV = (ym_costs_2035_PV - ym_costs_indstr2035) / emissions_indstrM_high_2035
CP2045h_green_PV = (ym_costs_2045_PV - ym_costs_indstr2045) / emissions_indstrM_high_2045

In [28]:
print("Carbon price compared to methanol with lower emission:", CPsql_PV,CP2035l_PV,CP2045l_PV)
print("Carbon price compared to methanol with higher emission:", CPsqh_PV,CP2035h_PV,CP2045h_PV)
print("Carbon price compared to methanol with lower emission (green case):", CPsql_green_PV,CP2035l_green_PV,CP2045l_green_PV)
print("Carbon price compared to methanol with higher emission (green_case):", CPsqh_green_PV,CP2035h_green_PV,CP2045h_green_PV)


print("CO2-equivalents:",emissions_per_tonne_sq,emissions_per_tonne_2035,emissions_per_tonne_2045,co2_indstr_low,co2_indstr_high)

Carbon price compared to methanol with lower emission: -325.14036907131225 964.5634152325094 464.1066580487805
Carbon price compared to methanol with higher emission: -430.66433571557195 549.1368863523417 332.31528082431015
Carbon price compared to methanol with lower emission (green case): 526.2539629268293 505.6544809756099 464.1066580487805
Carbon price compared to methanol with higher emission (green_case): 376.81474816625916 362.06485714285725 332.31528082431015
CO2-equivalents: 5.368015007122616 0.9753255207174446 0.0 2.05 2.863


In [29]:
# Create a dictionary to store the data
data_with_PV = {
    "Year": ["sq", "2035", "2045"],
    "Carbon Price Lower Emission": [CPsql_PV, CP2035l_PV, CP2045l_PV ],
    "Carbon Price Higher Emission": [CPsqh_PV, CP2035h_PV, CP2045h_PV],
    "Carbon Price Lower Emission (Green Case)": [CPsql_green_PV, CP2035l_green_PV, CP2045l_green_PV],
    "Carbon Price Higher Emission (Green Case)": [CPsqh_green_PV, CP2035h_green_PV, CP2045h_green_PV],
    "CO2 Equivalents": [emissions_per_tonne_sq, emissions_per_tonne_2035, emissions_per_tonne_2045],
    "CO2 Price [EUA]": [avg_EUA_sq, avg_EUA_2035, avg_EUA_2045]
}

# Convert the dictionary into a pandas DataFrame
df_values_with_PV = pd.DataFrame(data_with_PV)
df_values_with_PV

,Year,Carbon Price Lower Emission,Carbon Price Higher Emission,Carbon Price Lower Emission (Green Case),Carbon Price Higher Emission (Green Case),CO2 Equivalents,CO2 Price [EUA]
0,sq,-325.140369,-430.664336,526.253963,376.814748,5.368015,83.596502
1,2035,964.563415,549.136886,505.654481,362.064857,0.975326,133.666200
2,2045,464.106658,332.315281,464.106658,332.315281,0.000000,275.761800


In [30]:
#export
with pd.ExcelWriter(output_file_name) as writer:
    df_values_without_PV.to_excel(writer, sheet_name='values_BASE')
    df_values_with_PV.to_excel(writer, sheet_name='values_without_rev')